In [2]:
import pandas
import json, urllib.request
pandas.set_option('display.max_colwidth', None) # This will display the full content of the columns without truncation
pandas.set_option('display.max_rows', 200) # This will display all rows without truncation

In [3]:
EndpointsTable = pandas.read_html("https://api.erg.ic.ac.uk/AirQuality/help" )[0]
EndpointsTable # This prints the table


,Uri,Method,Description
0,/Annual/MonitoringObjective/GroupName={GroupName},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in XML format
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format
2,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in XML format
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in JSON format
4,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'LocalAuthorityId' and 'Year. Data returned in XML format
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'LocalAuthorityId' and 'Year. Data returned in JSON format
6,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate},GET,This returns the results of annual air quality objectives for monitoring sites for SiteCode and StartDate. Data returned in XML format
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,GET,This returns the results of annual air quality objectives for monitoring sites for SiteCode and StartDate. Data returned in JSON format
8,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year},GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'SiteCode' and 'Year. Data returned in XML format
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'SiteCode' and 'Year. Data returned in JSON format


In [4]:
EndpointsTable.dtypes # This , very conveniently , gives us the schema of the table

Uri            str
Method         str
Description    str
dtype: object

In [5]:
JsonEndpoints = EndpointsTable[EndpointsTable["Uri"].str.endswith("/Json")]
JsonEndpoints

,Uri,Method,Description
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName' and 'Year. Data returned in JSON format
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'LocalAuthorityId' and 'Year. Data returned in JSON format
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,GET,This returns the results of annual air quality objectives for monitoring sites for SiteCode and StartDate. Data returned in JSON format
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'SiteCode' and 'Year. Data returned in JSON format
11,/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json,GET,This returns the results of annual air quality report for monitoring site and Year. Data returned in JSON format
12,/Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json,GET,This returns the annualisation calculation!
13,/Annualiser/Sites/year={year}/species={species}/Json,GET,This returns sites which can be used in an annualisation calculation
15,/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json,GET,Returns the latest daily air quality monitoring indexes requested filtered by 'GroupName' and date. Currently the Group name would be 'London'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,GET,Returns the latest daily air quality monitoring indexes requested filtered by 'GroupName'. Currently the Group name would be 'London'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format


In [6]:
JsonEndpoints["Parameters"] = JsonEndpoints["Uri"].str.findall(r"\{(\w+)\}")
JsonEndpoints[["Uri", "Parameters"]] # This prints the table with only the Uri and Parameters columns

,Uri,Parameters
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,[GroupName]
3,/Annual/MonitoringObjective/GroupName={GroupName}/Year={Year}/Json,"[GroupName, Year]"
5,/Annual/MonitoringObjective/LocalAuthorityId={LocalAuthorityId}/Year={Year}/Json,"[LocalAuthorityId, Year]"
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,"[SiteCode, StartDate]"
9,/Annual/MonitoringObjective/SiteCode={SiteCode}/Year={Year}/Json,"[SiteCode, Year]"
11,/Annual/MonitoringReport/SiteCode={SiteCode}/Year={Year}/Json,"[SiteCode, Year]"
12,/Annualiser/Calculate/year={year}/site={site}/backA={backA}/backB={backB}/backC={backC}/backD={backD}/species={species}/mean={mean}/Json,"[year, site, backA, backB, backC, backD, species, mean]"
13,/Annualiser/Sites/year={year}/species={species}/Json,"[year, species]"
15,/Daily/MonitoringIndex/GroupName={GroupName}/Date={Date}/Json,"[GroupName, Date]"
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,[GroupName]


Lets take a look at the end points and what they return - a sensible order to is to first call the endpoints that require no parameters and seem like they would feed other important endpoints. We can start with 
/Information/Species/Json
/Information/Groups/Json


In [7]:
Species = "https://api.erg.ic.ac.uk/AirQuality/Information/Species/Json"
Groups =  "https://api.erg.ic.ac.uk/AirQuality/Information/Groups/Json"

In [8]:
SpeciesData = json.load(urllib.request.urlopen(Species))
SpeciesData # This prints the data of species

{'AirQualitySpecies': {'Species': [{'@SpeciesCode': 'CO',
    '@SpeciesName': 'Carbon Monoxide',
    '@Description': "Carbon Monoxide is a colourless, odourless poisonous gas produced by incomplete, or inefficient, combustion of fuel including 'cold' or badly tuned engines.",
    '@HealthEffect': 'The gas affects the transport of oxygen around the body by the blood. At very high levels, this can lead to a significant reduction in the supply of oxygen to the heart, particularly in people suffering from heart disease.',
    '@Link': 'http://www.londonair.org.uk/LondonAir/guide/WhatIsCO.aspx'},
   {'@SpeciesCode': 'NO2',
    '@SpeciesName': 'Nitrogen Dioxide',
    '@Description': 'Nitrogen oxides are formed during high temperature combustion processes from the oxidation of nitrogen in the air or fuel.',
    '@HealthEffect': 'Nitrogen Dioxide has several health impacts and includes general irritation to the eyes, irritation of the respiratory system and shortness of breath.',
    '@Link': 

In [9]:
print(SpeciesData.keys())
print(SpeciesData["AirQualitySpecies"].keys()) 
print(SpeciesData["AirQualitySpecies"]["Species"])
for dict in SpeciesData["AirQualitySpecies"]["Species"]:
    print(dict) # This prints the list of species

dict_keys(['AirQualitySpecies'])
dict_keys(['Species'])
[{'@SpeciesCode': 'CO', '@SpeciesName': 'Carbon Monoxide', '@Description': "Carbon Monoxide is a colourless, odourless poisonous gas produced by incomplete, or inefficient, combustion of fuel including 'cold' or badly tuned engines.", '@HealthEffect': 'The gas affects the transport of oxygen around the body by the blood. At very high levels, this can lead to a significant reduction in the supply of oxygen to the heart, particularly in people suffering from heart disease.', '@Link': 'http://www.londonair.org.uk/LondonAir/guide/WhatIsCO.aspx'}, {'@SpeciesCode': 'NO2', '@SpeciesName': 'Nitrogen Dioxide', '@Description': 'Nitrogen oxides are formed during high temperature combustion processes from the oxidation of nitrogen in the air or fuel.', '@HealthEffect': 'Nitrogen Dioxide has several health impacts and includes general irritation to the eyes, irritation of the respiratory system and shortness of breath.', '@Link': 'http://www.l

In [10]:
SpeciesTable = pandas.json_normalize(SpeciesData["AirQualitySpecies"]["Species"])
SpeciesTable # This prints the table of species

,@SpeciesCode,@SpeciesName,@Description,@HealthEffect,@Link
0,CO,Carbon Monoxide,"Carbon Monoxide is a colourless, odourless poisonous gas produced by incomplete, or inefficient, combustion of fuel including 'cold' or badly tuned engines.","The gas affects the transport of oxygen around the body by the blood. At very high levels, this can lead to a significant reduction in the supply of oxygen to the heart, particularly in people suffering from heart disease.",http://www.londonair.org.uk/LondonAir/guide/WhatIsCO.aspx
1,NO2,Nitrogen Dioxide,Nitrogen oxides are formed during high temperature combustion processes from the oxidation of nitrogen in the air or fuel.,"Nitrogen Dioxide has several health impacts and includes general irritation to the eyes, irritation of the respiratory system and shortness of breath.",http://www.londonair.org.uk/LondonAir/guide/WhatIsNO2.aspx
2,O3,Ozone,"Ozone is not directly emitted, but is formed by a complex set of reactions involving nitrogen oxides and hydrocarbons in the presence of sunlight.","Like nitrogen dioxide, high levels of ozone can irritate and inflame the lungs. It can also cause eye irritation, migraine and coughing.",http://www.londonair.org.uk/LondonAir/guide/WhatIsO3.aspx
3,PM10,PM10 Particulate,"Larger particles in the atmosphere are generally filtered in the nose and throat and do not cause problems, but particulate matter smaller than about 10 micrometers, referred to as PM10, can settle in the bronchi and lungs and cause health problems.","The effects of inhaling particulate matter have been widely studied in humans and animals and include asthma, respiratory problems and cardiovascular issues.",http://www.londonair.org.uk/LondonAir/guide/WhatIsPM.aspx
4,PM25,PM2.5 Particulate,"These are particles which are less than 2.5 micrometers (a thousandth of a millimetre) in diameter. Particles are made up of many different chemicals, including toxic metals and organic compounds, some of which are highly chemically reactive and researchers consider these to be involved in the toxicological effects of particles.",They are thought to have a greater effect on health than larger particles as they can be carried deeper into the lungs where they can cause inflammation and worsen heart and lung diseases.,http://www.londonair.org.uk/LondonAir/guide/WhatIsPM.aspx
5,SO2,Sulphur Dioxide,Sulphur Dioxide is produced when a material or fuel containing sulphur is burned. Pollution episodes only generally occur where there is widespread domestic use of coal or in the vicinity of coal or oil-fired power stations.,"Short-term exposure to high levels of sulphur dioxide may cause coughing, tightening of the chest and irritation of the lungs.",http://www.londonair.org.uk/LondonAir/guide/WhatIsSO2.aspx


In [11]:
GroupsData = json.load(urllib.request.urlopen(Groups))
GroupsData # This prints the data of groups

{'Groups': {'Group': [{'@GroupName': 'All',
    '@Description': 'All current network LAs',
    '@WebsiteURL': 'http://www.erg.kcl.ac.uk/'},
   {'@GroupName': 'AroundLondon',
    '@Description': 'Selection of boroughs around London',
    '@WebsiteURL': 'http://www.londonair.org.uk/'},
   {'@GroupName': 'CambridgeCity',
    '@Description': 'Demo group for tender',
    '@WebsiteURL': ''},
   {'@GroupName': 'Cambridgeshire',
    '@Description': 'Demo group for tender',
    '@WebsiteURL': ''},
   {'@GroupName': 'Essex',
    '@Description': 'All 14 councils in Essex',
    '@WebsiteURL': 'http://www.essexair.org.uk/'},
   {'@GroupName': 'Hampshire',
    '@Description': 'Various local authorities in Hampshire',
    '@WebsiteURL': 'http://www.hantsair.org.uk/'},
   {'@GroupName': 'Leicester',
    '@Description': 'Leicester Council',
    '@WebsiteURL': 'http://www.leicester.gov.uk/'},
   {'@GroupName': 'London',
    '@Description': 'All currently active boroughs of London',
    '@WebsiteURL': 'h

In [12]:
print(GroupsData.keys())
print(GroupsData["Groups"].keys())
print(GroupsData["Groups"]["Group"])
for dict in GroupsData["Groups"]["Group"]:
    print(dict) # This prints the list of groups

dict_keys(['Groups'])
dict_keys(['Group'])
[{'@GroupName': 'All', '@Description': 'All current network LAs', '@WebsiteURL': 'http://www.erg.kcl.ac.uk/'}, {'@GroupName': 'AroundLondon', '@Description': 'Selection of boroughs around London', '@WebsiteURL': 'http://www.londonair.org.uk/'}, {'@GroupName': 'CambridgeCity', '@Description': 'Demo group for tender', '@WebsiteURL': ''}, {'@GroupName': 'Cambridgeshire', '@Description': 'Demo group for tender', '@WebsiteURL': ''}, {'@GroupName': 'Essex', '@Description': 'All 14 councils in Essex', '@WebsiteURL': 'http://www.essexair.org.uk/'}, {'@GroupName': 'Hampshire', '@Description': 'Various local authorities in Hampshire', '@WebsiteURL': 'http://www.hantsair.org.uk/'}, {'@GroupName': 'Leicester', '@Description': 'Leicester Council', '@WebsiteURL': 'http://www.leicester.gov.uk/'}, {'@GroupName': 'London', '@Description': 'All currently active boroughs of London', '@WebsiteURL': 'http://www.londonair.org.uk/'}, {'@GroupName': 'LondonAirApps', 

In [13]:
GroupsTable = pandas.json_normalize(GroupsData["Groups"]["Group"])
GroupsTable # This prints the table of groups

,@GroupName,@Description,@WebsiteURL
0,All,All current network LAs,http://www.erg.kcl.ac.uk/
1,AroundLondon,Selection of boroughs around London,http://www.londonair.org.uk/
2,CambridgeCity,Demo group for tender,
3,Cambridgeshire,Demo group for tender,
4,Essex,All 14 councils in Essex,http://www.essexair.org.uk/
5,Hampshire,Various local authorities in Hampshire,http://www.hantsair.org.uk/
6,Leicester,Leicester Council,http://www.leicester.gov.uk/
7,London,All currently active boroughs of London,http://www.londonair.org.uk/
8,LondonAirApps,A selection of boroughs in and around London to be used by various iPhone/Android/Chrome/Widget apps. Boroughs who are not in LAQN not included but London Mean is (LAID 600),
9,Sussex,All 13 councils in Sussex,http://www.sussex-air.net/


Now we have groupnames and species codes , alongside dates since that is a free set - we can now build endpoints that have those parameters - lets check which endpoints we are now able to build after discovering the values for groups and species

In [14]:
Known = {"GroupName", "SpeciesCode", "StartDate", "EndDate"}
JsonEndpoints[JsonEndpoints["Parameters"].apply(lambda P: len(P) > 0 and set(P) <= Known)]

# .apply runs a function on every cell of a column.
# Here the function is a lambda whose input P is one cell 
# of the Parameters column, which is the list of parameters
# an endpoint needs. For each list, the lambda checks that
# it isn't empty and that every parameter in it is one we know.
# Converting the list to a set ignores order, and <= means 
# "is a subset of". The result is a column of True and False
# values, and indexing JsonEndpoints with it keeps only 
# the endpoints we can now call.


,Uri,Method,Description,Parameters
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format,[GroupName]
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,GET,Returns the latest daily air quality monitoring indexes requested filtered by 'GroupName'. Currently the Group name would be 'London'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format,[GroupName]
43,/Hourly/MonitoringIndex/GroupName={GroupName}/Json,GET,The returns the latest hourly air quality monitoring indexes requested filtered by 'GroupName'. Currently the Group name would be 'London'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format,[GroupName]
56,/Information/MonitoringLocalAuthority/GroupName={GroupName}/Json,GET,This lists the local authorities that have air quality monitoring within London. Currently the Group name would be 'London'. Data returned in JSON format,[GroupName]
58,/Information/MonitoringSites/GroupName={GroupName}/Json,GET,This lists the monitoring sites filtered by 'GroupName'. Currently the Group name would be 'London'. Data returned in JSON format,[GroupName]
60,/Information/MonitoringSiteSpecies/GroupName={GroupName}/Json,GET,This lists the monitoring sites and species filtered by 'GroupName'. Currently the Group name would be 'London'. Data returned in JSON format,[GroupName]
68,/Information/Species/SpeciesCode={SpeciesCode}/Json,GET,"This lists the commonly monitored pollutants filtered by 'SpeciesCode', along with their code, a description and the health effect. Data returned in JSON format",[SpeciesCode]


We can now see that 58 and 60 tell us sitecode - we want to call them so that we can construct parameteres with sitecode - we will choose the purer one , 58 - even though species code is relevant , its better not use a composite table where we can for the sake of cleanliness 
We need to now choose a value for our parameters GroupName and SpeciesCode so we can build monitoring sites - GroupName can be london for the sake of choosing a value that gives us as much data as possbile

In [15]:
MonitoringSites = "https://api.erg.ic.ac.uk/AirQuality/Information/MonitoringSites/GroupName=London/Json"
MonitoringSitesData = json.load(urllib.request.urlopen(MonitoringSites))
MonitoringSitesData # This prints the data of monitoring sites


{'Sites': {'Site': [{'@LocalAuthorityCode': '1',
    '@LocalAuthorityName': 'Barking and Dagenham',
    '@SiteCode': 'BG3',
    '@SiteName': 'Barking and Dagenham - North Street',
    '@SiteType': 'Kerbside',
    '@DateClosed': '2011-05-25 00:00:00',
    '@DateOpened': '2007-03-16 00:00:00',
    '@Latitude': '51.540444',
    '@Longitude': '0.074418',
    '@LatitudeWGS84': '6717454.5833',
    '@LongitudeWGS84': '8284.17386585',
    '@DisplayOffsetX': '0',
    '@DisplayOffsetY': '0',
    '@DataOwner': 'Barking and Dagenham',
    '@DataManager': "King's College London",
    '@SiteLink': 'http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG3'},
   {'@LocalAuthorityCode': '1',
    '@LocalAuthorityName': 'Barking and Dagenham',
    '@SiteCode': 'BG1',
    '@SiteName': 'Barking and Dagenham - Rush Green',
    '@SiteType': 'Suburban',
    '@DateClosed': '',
    '@DateOpened': '1999-11-02 00:00:00',
    '@Latitude': '51.563752',
    '@Longitude': '0.177891',
    '@LatitudeWGS84': '6

In [16]:
print(MonitoringSitesData.keys())
print(MonitoringSitesData["Sites"].keys())
print(MonitoringSitesData["Sites"]["Site"])
for dict in MonitoringSitesData["Sites"]["Site"]:
    print(dict) # This prints the list of monitoring sites


dict_keys(['Sites'])
dict_keys(['Site'])
[{'@LocalAuthorityCode': '1', '@LocalAuthorityName': 'Barking and Dagenham', '@SiteCode': 'BG3', '@SiteName': 'Barking and Dagenham - North Street', '@SiteType': 'Kerbside', '@DateClosed': '2011-05-25 00:00:00', '@DateOpened': '2007-03-16 00:00:00', '@Latitude': '51.540444', '@Longitude': '0.074418', '@LatitudeWGS84': '6717454.5833', '@LongitudeWGS84': '8284.17386585', '@DisplayOffsetX': '0', '@DisplayOffsetY': '0', '@DataOwner': 'Barking and Dagenham', '@DataManager': "King's College London", '@SiteLink': 'http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG3'}, {'@LocalAuthorityCode': '1', '@LocalAuthorityName': 'Barking and Dagenham', '@SiteCode': 'BG1', '@SiteName': 'Barking and Dagenham - Rush Green', '@SiteType': 'Suburban', '@DateClosed': '', '@DateOpened': '1999-11-02 00:00:00', '@Latitude': '51.563752', '@Longitude': '0.177891', '@LatitudeWGS84': '6721627.34498', '@LongitudeWGS84': '19802.7355367', '@DisplayOffsetX': '0', '@

In [17]:
MonitoringSitesTable = pandas.json_normalize(MonitoringSitesData["Sites"]["Site"])
MonitoringSitesTable # This prints the table of monitoring sites


,@LocalAuthorityCode,@LocalAuthorityName,@SiteCode,@SiteName,@SiteType,@DateClosed,@DateOpened,@Latitude,@Longitude,@LatitudeWGS84,@LongitudeWGS84,@DisplayOffsetX,@DisplayOffsetY,@DataOwner,@DataManager,@SiteLink
0,1,Barking and Dagenham,BG3,Barking and Dagenham - North Street,Kerbside,2011-05-25 00:00:00,2007-03-16 00:00:00,51.540444,0.074418,6717454.5833,8284.17386585,0,0,Barking and Dagenham,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG3
1,1,Barking and Dagenham,BG1,Barking and Dagenham - Rush Green,Suburban,,1999-11-02 00:00:00,51.563752,0.177891,6721627.34498,19802.7355367,0,0,Barking and Dagenham,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG1
2,1,Barking and Dagenham,BG2,Barking and Dagenham - Scrattons Farm,Suburban,,1999-10-17 00:00:00,51.529389,0.132857,6715476.18683,14789.5735883,0,0,Barking and Dagenham,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BG2
3,2,Barnet,BN2,Barnet - Finchley,Urban Background,2012-04-20 00:00:00,2000-08-09 13:00:00,51.591901,-0.205992,6726669.62886,-22930.9245475,0,0,Barnet,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BN2
4,2,Barnet,BN3,Barnet - Strawberry Vale,Urban Background,2002-05-15 00:00:00,2000-08-14 14:00:00,51.6008848453589,-0.172297542087178,6728279.54795,-19180.0746501,0,0,Barnet,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=BN3
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253,33,Westminster,WME,Westminster - Queens Park Gardens,Urban Background,2026-04-01 00:00:00,2025-07-03 00:00:00,51.5286629697555,-0.20875365618309,,,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=WME
254,33,Westminster,NB1,Westminster - Strand (Northbank BID),Roadside,2023-08-12 00:00:00,2015-04-28 00:00:00,51.5119701188277,-0.116713103528546,6712364.87549212,-12992.6543679369,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=NB1
255,33,Westminster,WM8,Westminster - Victoria,Urban Background,2016-04-04 00:00:00,2014-01-06 00:00:00,51.497066177578,-0.142438820740121,6709694.46727186,-15856.2169939846,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=WM8
256,33,Westminster,WM9,Westminster - Victoria (Victoria BID),Roadside,2017-09-20 00:00:00,2016-08-08 00:00:00,51.4977331829735,-0.14424140916201,6709818.70995547,-16057.0573104942,0,0,Westminster,King's College London,http://www.londonair.org.uk/london/asp/publicdetails.asp?site=WM9


In [18]:
Known.add("SiteCode")
JsonEndpoints[JsonEndpoints["Parameters"].apply(lambda P: len(P) > 0 and set(P) <= Known)]

,Uri,Method,Description,Parameters
1,/Annual/MonitoringObjective/GroupName={GroupName}/Json,GET,This returns the results of annual air quality objectives for monitoring sites filtered by 'GroupName'. Data returned in JSON format,[GroupName]
7,/Annual/MonitoringObjective/SiteCode={SiteCode}/StartDate={StartDate}/Json,GET,This returns the results of annual air quality objectives for monitoring sites for SiteCode and StartDate. Data returned in JSON format,"[SiteCode, StartDate]"
17,/Daily/MonitoringIndex/Latest/GroupName={GroupName}/Json,GET,Returns the latest daily air quality monitoring indexes requested filtered by 'GroupName'. Currently the Group name would be 'London'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format,[GroupName]
21,/Daily/MonitoringIndex/Latest/SiteCode={SiteCode}/Json,GET,The returns the latest daily air quality monitoring indexes requested filtered by 'SiteCode'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format,[SiteCode]
28,/Data/Site/SiteCode={SiteCode}/StartDate={StartDate}/EndDate={EndDate}/Json,GET,"This returns raw data based on 'SiteCode', 'StartDate', 'EndDate'. Default time period is 'hourly'. Data returned in JSON format","[SiteCode, StartDate, EndDate]"
32,/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json,GET,"This returns raw data based on 'SiteCode', 'SpeciesCode', 'StartDate', 'EndDate'. Default time period is 'hourly'. Data returned in JSON format","[SiteCode, SpeciesCode, StartDate, EndDate]"
35,/Data/Traffic/Site/SiteCode={SiteCode}/StartDate={StartDate}/EndDate={EndDate}/Json,GET,"This returns raw data based on 'SiteCode', 'StartDate', 'EndDate'. Data returned in JSON format from traffic counters.","[SiteCode, StartDate, EndDate]"
37,/Data/Wide/Site/SiteCode={SiteCode}/StartDate={StartDate}/EndDate={EndDate}/Json,GET,"This returns raw data based on 'SiteCode', 'StartDate', 'EndDate'. Data returned in JSON format with one entry per datetime (wide format).","[SiteCode, StartDate, EndDate]"
43,/Hourly/MonitoringIndex/GroupName={GroupName}/Json,GET,The returns the latest hourly air quality monitoring indexes requested filtered by 'GroupName'. Currently the Group name would be 'London'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format,[GroupName]
45,/Hourly/MonitoringIndex/SiteCode={SiteCode}/Json,GET,The returns the latest hourly air quality monitoring indexes requested filtered by 'SiteCode'. The health effect of these indexes can be found in the /Information/IndexHealthAdvice feed. Data returned in JSON format,[SiteCode]


In [ ]:
Parameters = {
              "SiteCode": "MY1", 
              "SpeciesCode": "NO2", 
              "StartDate": "2026-10-24",
              "EndDate": "2026-10-31"
              }

Template = "/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json"
Readings = "https://api.erg.ic.ac.uk/AirQuality" + Template.format(**Parameters)
ReadingsData = json.load(urllib.request.urlopen(Readings))
ReadingsData


{'RawAQData': {'@SiteCode': 'MY1',
  '@SpeciesCode': 'NO2',
  'Data': [{'@MeasurementDateGMT': '2026-10-24 00:00:00', '@Value': ''},
   {'@MeasurementDateGMT': '2026-10-24 01:00:00', '@Value': '36.5'},
   {'@MeasurementDateGMT': '2026-10-24 02:00:00', '@Value': '35.6'},
   {'@MeasurementDateGMT': '2026-10-24 03:00:00', '@Value': '34.9'},
   {'@MeasurementDateGMT': '2026-10-24 04:00:00', '@Value': '40'},
   {'@MeasurementDateGMT': '2026-10-24 05:00:00', '@Value': '41.8'},
   {'@MeasurementDateGMT': '2026-10-24 06:00:00', '@Value': '60.4'},
   {'@MeasurementDateGMT': '2026-10-24 07:00:00', '@Value': '55.7'},
   {'@MeasurementDateGMT': '2026-10-24 08:00:00', '@Value': '48'},
   {'@MeasurementDateGMT': '2026-10-24 09:00:00', '@Value': '47.1'},
   {'@MeasurementDateGMT': '2026-10-24 10:00:00', '@Value': '61.6'},
   {'@MeasurementDateGMT': '2026-10-24 11:00:00', '@Value': ''},
   {'@MeasurementDateGMT': '2026-10-24 12:00:00', '@Value': ''},
   {'@MeasurementDateGMT': '2026-10-24 13:00:00', '

In [38]:
print(ReadingsData.keys())
print(ReadingsData["RawAQData"].keys())
# This prints the keys of the RawAQData dictionary
print(ReadingsData["RawAQData"]["Data"])

dict_keys(['RawAQData'])
dict_keys(['@SiteCode', '@SpeciesCode', 'Data'])
[{'@MeasurementDateGMT': '2026-10-24 00:00:00', '@Value': ''}, {'@MeasurementDateGMT': '2026-10-24 01:00:00', '@Value': '36.5'}, {'@MeasurementDateGMT': '2026-10-24 02:00:00', '@Value': '35.6'}, {'@MeasurementDateGMT': '2026-10-24 03:00:00', '@Value': '34.9'}, {'@MeasurementDateGMT': '2026-10-24 04:00:00', '@Value': '40'}, {'@MeasurementDateGMT': '2026-10-24 05:00:00', '@Value': '41.8'}, {'@MeasurementDateGMT': '2026-10-24 06:00:00', '@Value': '60.4'}, {'@MeasurementDateGMT': '2026-10-24 07:00:00', '@Value': '55.7'}, {'@MeasurementDateGMT': '2026-10-24 08:00:00', '@Value': '48'}, {'@MeasurementDateGMT': '2026-10-24 09:00:00', '@Value': '47.1'}, {'@MeasurementDateGMT': '2026-10-24 10:00:00', '@Value': '61.6'}, {'@MeasurementDateGMT': '2026-10-24 11:00:00', '@Value': ''}, {'@MeasurementDateGMT': '2026-10-24 12:00:00', '@Value': ''}, {'@MeasurementDateGMT': '2026-10-24 13:00:00', '@Value': ''}, {'@MeasurementDateGMT

In [40]:
ReadingsTable = pandas.json_normalize(ReadingsData["RawAQData"]["Data"]).assign(**Parameters)
ReadingsTable # This prints the table of readings, with the parameters on every row


,@MeasurementDateGMT,@Value,SiteCode,SpeciesCode,StartDate,EndDate
0,2026-10-24 00:00:00,,MY1,NO2,2026-10-24,2026-10-31
1,2026-10-24 01:00:00,36.5,MY1,NO2,2026-10-24,2026-10-31
2,2026-10-24 02:00:00,35.6,MY1,NO2,2026-10-24,2026-10-31
3,2026-10-24 03:00:00,34.9,MY1,NO2,2026-10-24,2026-10-31
4,2026-10-24 04:00:00,40,MY1,NO2,2026-10-24,2026-10-31
5,2026-10-24 05:00:00,41.8,MY1,NO2,2026-10-24,2026-10-31
6,2026-10-24 06:00:00,60.4,MY1,NO2,2026-10-24,2026-10-31
7,2026-10-24 07:00:00,55.7,MY1,NO2,2026-10-24,2026-10-31
8,2026-10-24 08:00:00,48,MY1,NO2,2026-10-24,2026-10-31
9,2026-10-24 09:00:00,47.1,MY1,NO2,2026-10-24,2026-10-31


In [ ]:
# Parameters: {"SiteCode": "MY1", "SpeciesCode": "NO2", "StartDate": "YYYY-MM-DD", "EndDate": "YYYY-MM-DD"}
# EndDate itself is not included in the results.
def GetReadings(Parameters):
    Template = "/Data/SiteSpecies/SiteCode={SiteCode}/SpeciesCode={SpeciesCode}/StartDate={StartDate}/EndDate={EndDate}/Json"
    Url = "https://api.erg.ic.ac.uk/AirQuality" + Template.format(**Parameters)
    Data = json.load(urllib.request.urlopen(Url))
    return pandas.json_normalize(Data["RawAQData"]["Data"]).assign(**Parameters)

# Later we can validate inputs and shape with pydantic - for now the general idea is 
# to do things manually and abstact after




Now we can get some readings for different sites so we can compare them